# Spike V8: çevrimiçi deney (V7 sistemi × omurga S / M)

**Ne yapar:** `experiments/v2_v7.py`'yi `configs/v2_v8.yaml` ile, aynı hayvan için iki omurgayla koşar:
`--ckpt-stage day0` (S) ve `multiday` (M). Koşulların tamamı (A, R5, G7, D7 + ablasyonlar, B7, P7, C7,
beyincik ailesi, BOCPD kaynakları; ReFIT, BOCPD, ölü kanal, Degenhart açık). Sonra M − S tablosu.

- `PROTOKOL = "tune"`: AYAR oturumları, val, seed 100–102 (hiperparametre seçimi yalnız burada).
- `PROTOKOL = "test"`: TEST oturumları, test, seed 0–4 (rapor; ayar bittikten sonra, bir kez).
- `PROTOKOL = "ext"`: EK oturumlar, test, seed 0–4 (ayrı tablo).

**Önce:** `v8_egitim.ipynb` uygun `KIP` ile bitmiş olmalı (tune → ayar, test / ext → rapor). Çalışma zamanı
**GPU (L4)**; S ve M aynı GPU türünde koşmalı.

## 1. Ayarlar

In [ ]:
HAYVAN = "indy"
PROTOKOL = "tune"                  # tune | test | ext
EGITIM_AD = "v2_pretrain_v8"       # eğitim koşularının adı (readout varyantı: v2_pretrain_v8_ro)
DRIVE_KLASOR = "/content/drive/MyDrive/Spike_colab"
KOLLAR = ["day0", "multiday"]      # S, M
EK_ARGS = []                       # ör. ["--scenarios", "crossday"] (duman)
SPLIT = "val" if PROTOKOL == "tune" else "test"
SEEDLER = [100, 101, 102] if PROTOKOL == "tune" else [0, 1, 2, 3, 4]

## 2. GPU kontrolü

In [ ]:
!nvidia-smi
import sys, torch
print(sys.version)
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
assert torch.cuda.is_available(), "GPU yok: Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU (L4)"

## 3. Drive, kod ve önbellek

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, json, shutil, zipfile, pathlib, glob, subprocess, time
D = pathlib.Path(DRIVE_KLASOR)
assert (D / "spike_kod.zip").exists(), f"{D / 'spike_kod.zip'} yok: notebooks/README.md (V8) adım 1–3"
KOD = pathlib.Path("/content/Spike")
if KOD.exists():
    shutil.rmtree(KOD)
KOD.mkdir()
for f in ["spike_kod.zip", "spike_veri.zip"]:
    if (D / f).exists():
        with zipfile.ZipFile(D / f) as z:
            z.extractall(KOD)
os.chdir(KOD)
CACHE = KOD / "data/primate/cache"
CACHE.mkdir(parents=True, exist_ok=True)
DCACHE = D / "cache"                                   # v8_veri.ipynb'nin ürettiği önbellekler
DCACHE.mkdir(exist_ok=True)
for p in DCACHE.glob("*.pt"):
    if not (CACHE / p.name).exists():
        shutil.copy(p, CACHE / p.name)
surum = json.loads((KOD / "KOD_SURUMU.json").read_text(encoding="utf-8"))
print("kod:", surum["commit"], "(kirli)" if surum["dirty"] else "(temiz)", "| paket:", surum["paket_zamani"])
print("önbellek:", len(list(CACHE.glob("*.pt"))), "dosya")
ENV = {**os.environ, "PYTHONFAULTHANDLER": "1", "PYTHONIOENCODING": "utf-8"}

## 4. Bağımlılıklar ve hızlı test

In [ ]:
import torch as _t
_onceki = _t.__version__
!pip -q install pyyaml numpy pytest h5py neurobench==2.3.0
!python -c "import torch; from neurobench.metrics.workload import R2; print('neurobench tamam | torch', torch.__version__, '| CUDA', torch.cuda.is_available())"
print('kurulumdan önceki torch:', _onceki, '(üstteki satırla aynı olmalı; farklıysa Çalışma zamanı → Oturumu yeniden başlat)')
!python -m pytest tests/v2/test_v8_data.py tests/v2/test_v8_train.py -q -x

## 5. Çevrimiçi koşular

In [ ]:
EGITIM = sorted(glob.glob(str(D / "runs_v8" / f"*_{EGITIM_AD}_{HAYVAN}_seeds*")))
assert EGITIM, f"eğitim koşusu yok: {D / 'runs_v8'} / *_{EGITIM_AD}_{HAYVAN}_seeds*"
LOG_DIR = str(D / "runs_v8_online")
os.makedirs(LOG_DIR, exist_ok=True)
ISARET = pathlib.Path(LOG_DIR) / "_bitti"
ISARET.mkdir(exist_ok=True)
AD = f"v2_v8_{HAYVAN}" + ("" if EGITIM_AD == "v2_pretrain_v8" else "_ro")


def klasor(kol):
    suf = "" if kol == "day0" else f"_{kol}"
    return sorted(glob.glob(f"{LOG_DIR}/*_{AD}_{PROTOKOL}_{SPLIT}{suf}_seeds*"), reverse=True)


for kol in KOLLAR:
    isaret = ISARET / f"{AD}_{PROTOKOL}_{SPLIT}_{kol}"
    if isaret.exists():
        print("[atla]", kol)
        continue
    for deneme in (1, 2):
        cmd = ["python", "experiments/v2_v7.py", "--config", "configs/v2_v8.yaml", "--runs", *EGITIM,
               "--primates", HAYVAN, "--ckpt-stage", kol, "--protocol", PROTOKOL, "--split", SPLIT,
               "--seeds", *map(str, SEEDLER), "--set", f"experiment.log_dir={LOG_DIR}", f"experiment.name={AD}",
               *EK_ARGS]
        if PROTOKOL == "test":
            cmd.append("--electrode-crossday")
        log = pathlib.Path(LOG_DIR) / f"log_{AD}_{PROTOKOL}_{kol}_{deneme}.txt"
        t0 = time.time()
        with open(log, "w", encoding="utf-8") as f:
            rc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, env=ENV).returncode
        k = klasor(kol)
        if k:
            shutil.copy(KOD / "KOD_SURUMU.json", pathlib.Path(k[0]) / "KOD_SURUMU.json")
        print(f"{kol} deneme {deneme}: çıkış {rc}, {(time.time() - t0) / 60:.1f} dk → {k[0] if k else '-'}", flush=True)
        if rc == 0:
            isaret.touch()
            break
        print(log.read_text(encoding="utf-8")[-3000:])
    else:
        raise RuntimeError(f"{kol}: 2 denemede de başarısız")
print("BİTTİ:", HAYVAN, PROTOKOL, KOLLAR)

## 6. Tablo: M − S

`Omurga kolu: M − S` bölümü asıl V8 sonucudur. Tablo Drive'a (`v8_tablolar/`) da yazılır.

In [ ]:
S_, M_ = klasor("day0"), klasor("multiday")
assert S_ and M_, "iki kolun koşusu da gerekli"
cikis = D / "v8_tablolar" / f"{AD}_{PROTOKOL}_{SPLIT}"
subprocess.run(["python", "scripts/make_tables_v2.py", "--runs", S_[0], M_[0], "--backbones", "--main", "D7@M",
                "--claims", "D7@S", "A@M", "--extra", "A@S", "R5@M", "R5@S", "B7@M", "P7@M",
                "--title", f"Faz V8 {HAYVAN} {PROTOKOL}", "--out", str(cikis)], env=ENV, check=True)
print((cikis / "table_v6.md").read_text(encoding="utf-8")[:6000])

## 7. Bitince

Drive'ım → `Spike_colab/runs_v8_online/` ve `v8_tablolar/` klasörlerini bilgisayarında `Spike/runs/colab_v8/`
altına indir; Claude'a "V8 <hayvan> <protokol> bitti" de. Sonuçlar birlikte değerlendirildikten sonra dokümanlara işlenir.